# 🎬 Project 05 — Movie Recommendation System

**Description:** Build a content-based movie recommender using TF-IDF vectorization and cosine similarity.  
**Learning Goals:**
- Understand how content-based filtering works
- Learn what TF-IDF is and why it's used in NLP
- Compute and interpret cosine similarity scores
- Visualize recommendations with matplotlib

**Estimated Time:** 45–60 minutes  
**Difficulty:** ⭐⭐ Beginner–Intermediate

## Step 1 — Install & Import Libraries

We need:
- **pandas** — to organise the movie dataset in a table
- **scikit-learn** — for TF-IDF and cosine similarity
- **matplotlib** — to draw a bar chart of results

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

print('Libraries loaded successfully!')

## Step 2 — Create the Movie Dataset

Each movie is described by a short string of **genre/keyword tags**.  
This is the "content" the recommender will use to find similar films.

In [ ]:
MOVIES = [
    {"title": "The Dark Knight",        "tags": "action crime thriller superhero batman villain"},
    {"title": "Inception",              "tags": "sci-fi thriller dream heist mind-bending"},
    {"title": "Interstellar",           "tags": "sci-fi space adventure drama time-travel"},
    {"title": "The Matrix",             "tags": "sci-fi action thriller virtual-reality dystopia"},
    {"title": "Avengers: Endgame",      "tags": "action superhero marvel adventure"},
    {"title": "The Godfather",          "tags": "crime drama mafia family classic"},
    {"title": "Pulp Fiction",           "tags": "crime thriller drama cult non-linear"},
    {"title": "Forrest Gump",           "tags": "drama romance comedy history inspirational"},
    {"title": "The Lion King",          "tags": "animation family drama adventure musical"},
    {"title": "Toy Story",              "tags": "animation family comedy adventure friendship"},
    {"title": "Finding Nemo",           "tags": "animation family adventure comedy ocean"},
    {"title": "Up",                     "tags": "animation family adventure drama friendship"},
    {"title": "Shrek",                  "tags": "animation comedy family fairy-tale adventure"},
    {"title": "The Avengers",           "tags": "action superhero marvel adventure team"},
    {"title": "Iron Man",               "tags": "action superhero marvel tech billionaire"},
    {"title": "Spider-Man: No Way Home","tags": "action superhero marvel spider-man multiverse"},
    {"title": "Doctor Strange",         "tags": "action superhero marvel magic multiverse"},
    {"title": "Guardians of the Galaxy","tags": "action sci-fi marvel comedy adventure space"},
    {"title": "Gravity",                "tags": "sci-fi space thriller survival drama"},
    {"title": "The Martian",            "tags": "sci-fi space adventure survival drama comedy"},
    {"title": "Mad Max: Fury Road",     "tags": "action sci-fi thriller post-apocalyptic survival"},
    {"title": "John Wick",              "tags": "action thriller crime assassin revenge"},
    {"title": "Taken",                  "tags": "action thriller crime rescue revenge"},
    {"title": "The Silence of the Lambs","tags": "thriller crime horror psychological detective"},
    {"title": "Se7en",                  "tags": "thriller crime mystery detective dark"},
    {"title": "Fight Club",             "tags": "drama thriller cult psychology identity"},
    {"title": "The Shawshank Redemption","tags": "drama crime hope friendship classic"},
    {"title": "Schindler's List",       "tags": "drama history war hope tragedy"},
    {"title": "Titanic",                "tags": "romance drama history tragedy disaster"},
    {"title": "La La Land",             "tags": "romance drama musical comedy ambition"},
]

df = pd.DataFrame(MOVIES)
print(f'Dataset: {len(df)} movies')
df.head(10)

## Step 3 — TF-IDF Vectorization

### What is TF-IDF?
**TF** (Term Frequency) — how often a word appears in *this* movie's tags.  
**IDF** (Inverse Document Frequency) — penalises words that appear in *many* movies (e.g. "action") so they carry less weight.

The result is a **numeric vector** for each movie.  
Movies with similar tag vectors are likely similar films.

In [ ]:
# Create the TF-IDF vectorizer
tfidf = TfidfVectorizer(stop_words='english')   # ignores common English words like 'the', 'a'

# Fit on the tags column and transform into a matrix
tfidf_matrix = tfidf.fit_transform(df['tags'])

print(f'TF-IDF matrix shape: {tfidf_matrix.shape}')
print(f'  → {tfidf_matrix.shape[0]} movies x {tfidf_matrix.shape[1]} unique keywords')

# Peek at the vocabulary (first 20 words)
vocab = tfidf.get_feature_names_out()
print(f'\nSample vocabulary (first 20): {list(vocab[:20])}')

## Step 4 — Compute Cosine Similarity

**Cosine similarity** measures the angle between two vectors:
- **1.0** → identical direction (very similar)
- **0.0** → completely different

We compute this for every pair of movies at once — producing a 30×30 matrix.

In [ ]:
similarity = cosine_similarity(tfidf_matrix, tfidf_matrix)

print(f'Similarity matrix shape: {similarity.shape}')
print(f'Diagonal value (movie vs itself): {similarity[0][0]:.2f}  ← always 1.0')

# Show as a DataFrame for readability
sim_df = pd.DataFrame(similarity, index=df['title'], columns=df['title'])
sim_df.iloc[:6, :6].round(2)

## Step 5 — Recommendation Function

In [ ]:
def get_recommendations(title, top_n=5):
    """Return the top-N most similar movies for a given title."""
    matches = df[df['title'].str.lower() == title.lower()]
    if matches.empty:
        return f"Movie '{title}' not found in dataset."

    movie_idx = matches.index[0]

    # Get scores and sort descending; exclude the movie itself (index 0 after sort)
    scores = sorted(enumerate(similarity[movie_idx]), key=lambda x: x[1], reverse=True)
    top_scores = scores[1: top_n + 1]

    results = []
    for idx, score in top_scores:
        results.append({
            'Recommended Movie': df.iloc[idx]['title'],
            'Tags':              df.iloc[idx]['tags'],
            'Similarity Score':  round(score, 4)
        })

    return pd.DataFrame(results)

# Test it!
print('=== Recommendations for: Inception ===')
get_recommendations('Inception')

In [ ]:
print('=== Recommendations for: The Lion King ===')
get_recommendations('The Lion King')

In [ ]:
print('=== Recommendations for: John Wick ===')
get_recommendations('John Wick')

## Step 6 — Visualize Similarity Scores

In [ ]:
def plot_recommendations(title, top_n=5):
    recs = get_recommendations(title, top_n)
    if isinstance(recs, str):
        print(recs)
        return

    titles = recs['Recommended Movie'].tolist()[::-1]
    scores = recs['Similarity Score'].tolist()[::-1]

    fig, ax = plt.subplots(figsize=(9, 5))
    colors = plt.cm.Blues(np.linspace(0.4, 0.85, len(titles)))
    bars = ax.barh(titles, scores, color=colors, edgecolor='steelblue', linewidth=0.6)

    for bar, score in zip(bars, scores):
        ax.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height() / 2,
                f'{score:.3f}', va='center', fontsize=9)

    ax.set_xlabel('Cosine Similarity Score')
    ax.set_title(f'Top {top_n} Recommendations for "{title}"', fontweight='bold')
    ax.set_xlim(0, max(scores) * 1.3)
    ax.spines[['top', 'right']].set_visible(False)
    plt.tight_layout()
    plt.savefig('recommendations.png', dpi=150)
    plt.show()

plot_recommendations('Inception')

## Step 7 — Try It Yourself 🚀

Modify the cell below to get recommendations for any movie in the list.  
You can also try:
1. Changing `top_n` to get more or fewer results
2. Adding a new movie to the `MOVIES` list and rebuilding the matrix
3. Editing the `tags` to see how recommendations change

In [ ]:
# 🎯 Change this to any movie title from the dataset
my_movie = 'Interstellar'
top_n    = 5

recs = get_recommendations(my_movie, top_n)
print(f'Top {top_n} movies similar to "{my_movie}":\n')
print(recs.to_string(index=False))

## Summary — Concepts Learned

| Concept | Plain-English Explanation |
|---------|---------------------------|
| **Content-Based Filtering** | Recommend items similar to what you already like, based on their features/tags |
| **TF-IDF** | Convert words to numbers; rare words get higher weight than common ones |
| **Cosine Similarity** | Measure the angle between two vectors — score close to 1 means very similar |
| **Vectorization** | Turning text into a numeric array so a computer can do math on it |
| **Pandas DataFrame** | A spreadsheet-like table in Python, great for storing and slicing data |
| **Collaborative Filtering** | (Next step!) Uses *user ratings* instead of item features — powers Netflix & Spotify |